## Step 1 — Inspect images and map them to ASINs

In [1]:
# ============================================================
# Step 1.1: Environment and project paths
# ============================================================

from google.colab import drive
from pathlib import Path

import hashlib
import re
import zipfile

import pandas as pd
from PIL import Image, UnidentifiedImageError


drive.mount("/content/drive")

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/"
    "Amazon-Marketplace-Product-Intelligence-Platform-BJIT"
)

IMAGE_ZIP_PATH = (
    PROJECT_ROOT
    / "data"
    / "images"
    / "thumbnail_images.zip"
)

COLAB_IMAGE_DIR = Path(
    "/content/thumbnail_images"
)

INTERIM_DIR = (
    PROJECT_ROOT
    / "data"
    / "interim"
)

REPORT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "thumbnail_grouping"
)

INTERIM_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Project exists:", PROJECT_ROOT.exists())
print("Image ZIP exists:", IMAGE_ZIP_PATH.exists())

Mounted at /content/drive
Project exists: True
Image ZIP exists: True


## Cell 2 — Locate and load the product dataset

This automatically uses the cleaned recommendation dataset if available. It falls back to other possible dataset locations.

In [2]:
# ============================================================
# Step 1.2: Locate and load product data
# ============================================================

DATASET_CANDIDATES = [
    (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "products_recommendation_clean.csv"
    ),
    (
        PROJECT_ROOT
        / "data"
        / "raw"
        / "amazon_products.csv"
    ),
    (
        PROJECT_ROOT
        / "data"
        / "amazon_products.csv"
    ),
]

DATASET_PATH = next(
    (
        path
        for path in DATASET_CANDIDATES
        if path.exists()
    ),
    None,
)

if DATASET_PATH is None:
    raise FileNotFoundError(
        "Product dataset was not found. Checked:\n"
        + "\n".join(
            str(path)
            for path in DATASET_CANDIDATES
        )
    )

products = pd.read_csv(
    DATASET_PATH,
    dtype={"asin": str},
)

if "asin" not in products.columns:
    raise ValueError(
        "The product dataset does not contain an "
        "'asin' column."
    )

products["asin"] = (
    products["asin"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.upper()
)

products = products[
    products["asin"].ne("")
].copy()

product_index = (
    products
    .drop_duplicates(
        subset="asin",
        keep="first",
    )
    .reset_index(drop=True)
)

print("Dataset:", DATASET_PATH)
print("Dataset rows:", len(products))
print(
    "Unique ASINs:",
    product_index["asin"].nunique(),
)

Dataset: /content/drive/MyDrive/Amazon-Marketplace-Product-Intelligence-Platform-BJIT/data/processed/products_recommendation_clean.csv
Dataset rows: 981
Unique ASINs: 981


## Cell 3 — Extract the existing image archive

This extracts the ZIP into Colab’s temporary fast storage.

In [3]:
# ============================================================
# Step 1.3: Extract thumbnail archive
# ============================================================

if not IMAGE_ZIP_PATH.exists():
    raise FileNotFoundError(
        f"Image ZIP not found: {IMAGE_ZIP_PATH}"
    )

COLAB_IMAGE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

with zipfile.ZipFile(
    IMAGE_ZIP_PATH,
    "r",
) as zip_file:
    zip_file.extractall(
        COLAB_IMAGE_DIR
    )

SUPPORTED_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".webp",
    ".bmp",
    ".tif",
    ".tiff",
}

image_paths = sorted(
    [
        path
        for path in COLAB_IMAGE_DIR.rglob("*")
        if (
            path.is_file()
            and path.suffix.lower()
            in SUPPORTED_EXTENSIONS
        )
    ]
)

if not image_paths:
    raise ValueError(
        "No supported image files were found "
        "inside the ZIP."
    )

print("Extracted image files:", len(image_paths))
print("Extraction directory:", COLAB_IMAGE_DIR)

print("\nFirst 10 filenames:")

for image_path in image_paths[:10]:
    print(image_path.name)

Extracted image files: 981
Extraction directory: /content/thumbnail_images

First 10 filenames:
DSLR camera_B0012OGF6Q_amazon_us.jpg
DSLR camera_B004U4UHTQ_amazon_us.jpg
DSLR camera_B007VGGFZU_amazon_us.jpg
DSLR camera_B0101RW3QK_amazon_us.jpg
DSLR camera_B06Y5RTN1T_amazon_us.jpg
DSLR camera_B06Y5ZCFHX_amazon_us.jpg
DSLR camera_B07B45D8WV_amazon_us.jpg
DSLR camera_B07PCSKGP7_amazon_us.jpg
DSLR camera_B086TTTZR5_amazon_us.jpg
DSLR camera_B09ZRN1N3Z_amazon_us.jpg


## Cell 4 — Map filenames and inspect image validity

This cell maps every filename to a known ASIN, reads its dimensions and calculates an exact-image hash.

In [4]:
# ============================================================
# Step 1.4: Map images and validate files
# ============================================================

known_asins = set(
    product_index["asin"].tolist()
)


def find_asin_in_filename(image_path):
    stem = image_path.stem.upper()

    # Exact filename, such as B00005IBX9.jpg
    if stem in known_asins:
        return stem, "exact_filename"

    # Find isolated 10-character ASIN tokens.
    candidates = re.findall(
        r"(?<![A-Z0-9])"
        r"([A-Z0-9]{10})"
        r"(?![A-Z0-9])",
        stem,
    )

    valid_candidates = [
        candidate
        for candidate in candidates
        if candidate in known_asins
    ]

    if len(valid_candidates) == 1:
        return (
            valid_candidates[0],
            "filename_token",
        )

    # Fallback for filenames containing prefixes or suffixes.
    embedded_matches = [
        asin
        for asin in known_asins
        if asin in stem
    ]

    if len(embedded_matches) == 1:
        return (
            embedded_matches[0],
            "embedded_asin",
        )

    if len(embedded_matches) > 1:
        return "", "ambiguous"

    return "", "unmatched"


def calculate_sha256(image_path):
    digest = hashlib.sha256()

    with open(image_path, "rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


inspection_rows = []

for number, image_path in enumerate(
    image_paths,
    start=1,
):
    asin, mapping_status = (
        find_asin_in_filename(image_path)
    )

    image_status = "valid"
    error_message = ""

    width = None
    height = None
    image_format = ""
    image_mode = ""

    try:
        with Image.open(image_path) as image:
            image.load()

            width, height = image.size
            image_format = (
                image.format or ""
            )
            image_mode = image.mode

    except (
        UnidentifiedImageError,
        OSError,
        ValueError,
    ) as error:
        image_status = "corrupt"
        error_message = str(error)

    try:
        sha256 = calculate_sha256(
            image_path
        )
    except OSError as error:
        sha256 = ""
        image_status = "unreadable"
        error_message = str(error)

    inspection_rows.append({
        "asin": asin,
        "filename": image_path.name,
        "relative_path": str(
            image_path.relative_to(
                COLAB_IMAGE_DIR
            )
        ),
        "extension": (
            image_path.suffix.lower()
        ),
        "file_size_kb": round(
            image_path.stat().st_size
            / 1024,
            2,
        ),
        "width": width,
        "height": height,
        "aspect_ratio": (
            round(width / height, 4)
            if width and height
            else None
        ),
        "image_format": image_format,
        "image_mode": image_mode,
        "sha256": sha256,
        "mapping_status": mapping_status,
        "image_status": image_status,
        "error_message": error_message,
    })

    if (
        number % 100 == 0
        or number == len(image_paths)
    ):
        print(
            f"Inspected {number}/"
            f"{len(image_paths)} images"
        )


thumbnail_inventory = pd.DataFrame(
    inspection_rows
)

print("Image inspection completed.")

Inspected 100/981 images
Inspected 200/981 images
Inspected 300/981 images
Inspected 400/981 images
Inspected 500/981 images
Inspected 600/981 images
Inspected 700/981 images
Inspected 800/981 images
Inspected 900/981 images
Inspected 981/981 images
Image inspection completed.


## Cell 5 — Add product information and calculate the summary

This produces the important Step 1 results.

In [5]:
# ============================================================
# Step 1.5: Inspection summary
# ============================================================

metadata_columns = [
    column
    for column in [
        "asin",
        "title",
        "image_url",
        "product_url",
    ]
    if column in product_index.columns
]

thumbnail_inventory = (
    thumbnail_inventory
    .merge(
        product_index[
            metadata_columns
        ],
        on="asin",
        how="left",
    )
)

valid_mapped_images = (
    thumbnail_inventory[
        thumbnail_inventory[
            "image_status"
        ].eq("valid")
        & thumbnail_inventory[
            "asin"
        ].ne("")
    ]
)

asins_with_valid_images = set(
    valid_mapped_images["asin"]
)

missing_asins = sorted(
    known_asins
    - asins_with_valid_images
)

images_per_asin = (
    valid_mapped_images
    .groupby("asin")
    .size()
)

asins_with_multiple_images = (
    images_per_asin[
        images_per_asin > 1
    ]
)

valid_hash_counts = (
    thumbnail_inventory[
        thumbnail_inventory[
            "image_status"
        ].eq("valid")
        & thumbnail_inventory[
            "sha256"
        ].ne("")
    ]
    .groupby("sha256")
    .size()
)

duplicate_hash_groups = (
    valid_hash_counts[
        valid_hash_counts > 1
    ]
)

inspection_summary = pd.DataFrame({
    "metric": [
        "Product rows",
        "Unique product ASINs",
        "Extracted image files",
        "Valid image files",
        "Corrupt/unreadable image files",
        "Matched image files",
        "Unmatched image files",
        "Ambiguous filename mappings",
        "Unique ASINs with valid images",
        "Products without valid images",
        "ASINs with multiple images",
        "Exact duplicate image groups",
    ],
    "value": [
        len(products),
        len(known_asins),
        len(thumbnail_inventory),
        int(
            thumbnail_inventory[
                "image_status"
            ].eq("valid").sum()
        ),
        int(
            thumbnail_inventory[
                "image_status"
            ].ne("valid").sum()
        ),
        int(
            thumbnail_inventory[
                "asin"
            ].ne("").sum()
        ),
        int(
            thumbnail_inventory[
                "mapping_status"
            ].eq("unmatched").sum()
        ),
        int(
            thumbnail_inventory[
                "mapping_status"
            ].eq("ambiguous").sum()
        ),
        len(asins_with_valid_images),
        len(missing_asins),
        len(asins_with_multiple_images),
        len(duplicate_hash_groups),
    ],
})

display(inspection_summary)

,metric,value
0,Product rows,981
1,Unique product ASINs,981
2,Extracted image files,981
3,Valid image files,981
4,Corrupt/unreadable image files,0
5,Matched image files,981
6,Unmatched image files,0
7,Ambiguous filename mappings,0
8,Unique ASINs with valid images,981
9,Products without valid images,0


## Cell 6 — Review distributions and problematic files

This shows whether the mapping is suitable before preprocessing.

In [6]:
# ============================================================
# Step 1.6: Review inspection results
# ============================================================

print("Image extensions")

display(
    thumbnail_inventory[
        "extension"
    ]
    .value_counts()
    .rename_axis("extension")
    .reset_index(name="count")
)

print("\nMapping status")

display(
    thumbnail_inventory[
        "mapping_status"
    ]
    .value_counts()
    .rename_axis("mapping_status")
    .reset_index(name="count")
)

print("\nSample mapped images")

display(
    thumbnail_inventory[
        [
            "asin",
            "filename",
            "mapping_status",
            "image_status",
            "width",
            "height",
            "title",
        ]
    ].head(10)
)

print("\nProducts without valid images")

missing_products = (
    product_index[
        product_index[
            "asin"
        ].isin(missing_asins)
    ]
)

display(
    missing_products[
        [
            column
            for column in [
                "asin",
                "title",
                "image_url",
            ]
            if column
            in missing_products.columns
        ]
    ].head(20)
)

print("\nASINs with multiple images")

display(
    asins_with_multiple_images
    .rename("image_count")
    .reset_index()
    .head(20)
)

Image extensions


,extension,count
0,.jpg,981



Mapping status


,mapping_status,count
0,filename_token,981



Sample mapped images


,asin,filename,mapping_status,image_status,width,height,title
0,B0012OGF6Q,DSLR camera_B0012OGF6Q_amazon_us.jpg,filename_token,valid,1100,1025,Nikon D60 DSLR Camera with 18-55mm f/3.5-5.6G ...
1,B004U4UHTQ,DSLR camera_B004U4UHTQ_amazon_us.jpg,filename_token,valid,1500,1500,Nikon D5100 16.2MP Digital SLR Camera & 18-55m...
2,B007VGGFZU,DSLR camera_B007VGGFZU_amazon_us.jpg,filename_token,valid,1206,1500,Nikon D3200 24.2 MP CMOS Digital SLR with 18-5...
3,B0101RW3QK,DSLR camera_B0101RW3QK_amazon_us.jpg,filename_token,valid,1500,1181,Canon EOS Rebel T7 DSLR Camera Double Zoom Len...
4,B06Y5RTN1T,DSLR camera_B06Y5RTN1T_amazon_us.jpg,filename_token,valid,1500,1500,Nikon D7500 20.9MP DSLR Camera with AF-S DX NI...
5,B06Y5ZCFHX,DSLR camera_B06Y5ZCFHX_amazon_us.jpg,filename_token,valid,1500,1500,Nikon D7500 DX-Format Digital SLR Body
6,B07B45D8WV,DSLR camera_B07B45D8WV_amazon_us.jpg,filename_token,valid,1500,1500,Sony Alpha a7 III Mirrorless Camera with 28-70...
7,B07PCSKGP7,DSLR camera_B07PCSKGP7_amazon_us.jpg,filename_token,valid,1298,1300,Canon EOS Rebel T7 DSLR Camera with EF-S 18-55...
8,B086TTTZR5,DSLR camera_B086TTTZR5_amazon_us.jpg,filename_token,valid,1500,1500,Canon EOS RP Full-Frame Mirrorless Interchange...
9,B09ZRN1N3Z,DSLR camera_B09ZRN1N3Z_amazon_us.jpg,filename_token,valid,1500,1500,"KODAK PIXPRO FZ55 16MP Compact Digital Camera,..."



Products without valid images


,asin,title,image_url



ASINs with multiple images


,asin,image_count


# Cell 7 — Save Step 1 outputs

This saves the results so they remain available after the Colab session ends

In [7]:
# ============================================================
# Step 1.7: Save image inspection outputs
# ============================================================

INVENTORY_PATH = (
    INTERIM_DIR
    / "thumbnail_image_inventory.csv"
)

SUMMARY_PATH = (
    REPORT_DIR
    / "thumbnail_inspection_summary.csv"
)

MISSING_PRODUCTS_PATH = (
    REPORT_DIR
    / "thumbnail_missing_products.csv"
)

MULTIPLE_IMAGES_PATH = (
    REPORT_DIR
    / "thumbnail_multiple_images.csv"
)

thumbnail_inventory.to_csv(
    INVENTORY_PATH,
    index=False,
    encoding="utf-8-sig",
)

inspection_summary.to_csv(
    SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig",
)

missing_products.to_csv(
    MISSING_PRODUCTS_PATH,
    index=False,
    encoding="utf-8-sig",
)

(
    asins_with_multiple_images
    .rename("image_count")
    .reset_index()
    .to_csv(
        MULTIPLE_IMAGES_PATH,
        index=False,
        encoding="utf-8-sig",
    )
)

print("Step 1 completed successfully.")
print("Inventory:", INVENTORY_PATH)
print("Summary:", SUMMARY_PATH)
print(
    "Missing products:",
    MISSING_PRODUCTS_PATH,
)
print(
    "Multiple images:",
    MULTIPLE_IMAGES_PATH,
)

Step 1 completed successfully.
Inventory: /content/drive/MyDrive/Amazon-Marketplace-Product-Intelligence-Platform-BJIT/data/interim/thumbnail_image_inventory.csv
Summary: /content/drive/MyDrive/Amazon-Marketplace-Product-Intelligence-Platform-BJIT/reports/thumbnail_grouping/thumbnail_inspection_summary.csv
Missing products: /content/drive/MyDrive/Amazon-Marketplace-Product-Intelligence-Platform-BJIT/reports/thumbnail_grouping/thumbnail_missing_products.csv
Multiple images: /content/drive/MyDrive/Amazon-Marketplace-Product-Intelligence-Platform-BJIT/reports/thumbnail_grouping/thumbnail_multiple_images.csv
